In [1]:
import os

In [2]:
%pwd

'/Users/admin/PycharmProjects/AnimalsDetectionMLOPS/notebooks'

In [3]:
os.chdir("../")

In [5]:
from dataclasses import dataclass
from pathlib import Path

import torch
from torch.utils.tensorboard import SummaryWriter

from src.cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.cnnClassifier.utils.common import read_yaml, create_directories


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    """Configuration for preparing training callbacks."""
    root_dir: Path
    checkpoint_model_filepath: Path
    tensorboard_log_dir: Path


class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_callbacks_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        create_directories([config.root_dir])
        return PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath),
            tensorboard_log_dir=Path(config.tensorboard_root_log_dir),
        )


class CheckpointCallback:
    """Callback that saves model state dict per epoch."""

    def __init__(self, filepath: Path):
        self.filepath = filepath

    def save(self, model, epoch: int):
        torch.save(model.state_dict(), str(self.filepath).format(epoch=epoch))


class PrepareCallbacks:
    """Factory for preparing TensorBoard and checkpoint callbacks."""

    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config
        create_directories([
            self.config.root_dir,
            self.config.tensorboard_log_dir.parent,
            self.config.checkpoint_model_filepath.parent,
        ])

    def get_tensorboard_callback(self) -> SummaryWriter:
        """Return a torch SummaryWriter pointing to the tensorboard log dir."""
        return SummaryWriter(log_dir=str(self.config.tensorboard_log_dir))

    def get_checkpoint_callback(self) -> CheckpointCallback:
        """Return a checkpoint callback that saves model state dict per epoch."""
        return CheckpointCallback(filepath=self.config.checkpoint_model_filepath)

In [6]:
try:
    config_manager = ConfigurationManager()
    prepare_callbacks_config = config_manager.get_prepare_callbacks_config()
    prepare_callbacks = PrepareCallbacks(config=prepare_callbacks_config)

    tensorboard_callback = prepare_callbacks.get_tensorboard_callback()
    checkpoint_callback = prepare_callbacks.get_checkpoint_callback()

    print(f"TensorBoard log dir: {prepare_callbacks_config.tensorboard_log_dir}")
    print(f"Checkpoint model filepath: {prepare_callbacks_config.checkpoint_model_filepath}")
except Exception as e:
    print(f"Error occurred: {e}")

[2026-01-22 13:06:45,129: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-01-22 13:06:45,131: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-22 13:06:45,131: INFO: common: created directory at: artifacts]
[2026-01-22 13:06:45,132: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-01-22 13:06:45,132: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-01-22 13:06:45,133: INFO: common: created directory at: artifacts/prepare_callbacks]
[2026-01-22 13:06:45,134: INFO: common: created directory at: artifacts/prepare_callbacks/checkpoint_dir]
TensorBoard log dir: artifacts/prepare_callbacks/tensorboard_log_dir
Checkpoint model filepath: artifacts/prepare_callbacks/checkpoint_dir/model.h5
